# NumPy + pandas 

**Prerequisites:** None beyond basic Python syntax.

**References (official docs):**
- NumPy Absolute Beginners: https://numpy.org/doc/stable/user/absolute_beginners.html  
- pandas “10 Minutes to pandas”: https://pandas.pydata.org/docs/user_guide/10min.html  
- pandas User Guide: https://pandas.pydata.org/docs/user_guide/index.html  


## 0. Setup

Run this cell first.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# If you're in Jupyter, plots will show inline:
%matplotlib inline

## 1. NumPy basics: arrays, vectorization, broadcasting

NumPy arrays are **typed**, **vectorized** containers for numerical data.  
They are the foundation for most scientific Python tools.

### 1.1 Creating arrays

In [ ]:
x = np.array([1, 2, 3, 4])
x, x.dtype, x.shape

In [ ]:
np.zeros(5), np.ones(5), np.linspace(0, 1, 5)

### 1.2 Vectorized operations (no loops)

Operations apply elementwise.

In [ ]:
a = np.array([1, 2, 3])
b = np.array([10, 20, 30])

a + b, a * b

### 1.3 Broadcasting

A scalar can be combined with an array (and more general shapes can broadcast too).

In [ ]:
a + 5

### Exercise 1 (NumPy warm-up)

Create an array of **100** normally distributed daily returns with mean **1%** and daily standard deviation **5%**.

Compute:
1. sample mean  
2. sample std  
3. cumulative simple return (growth of $1 under simple compounding)  
4. cumulative log return (growth of $1 under log compounding)

In [ ]:
# naive python
np.random.seed(0)  # so everyone has the same numbers
n = 100
r = np.random.normal(0.01, 0.05, n)
sum_r_manual = 0
num_r_manual = 0
sum_r2_manual = 0
sum_logret_manual = 0
for i in range(n):
    sum_r_manual = sum_r_manual + r[i]
    sum_r2_manual = sum_r2_manual + r[i] ** 2
    num_r_manual = num_r_manual + 1
    sum_logret_manual = sum_logret_manual + np.log(1+r[i])

mean_r_manual = sum_r_manual / num_r_manual
print (f'mean_r_manual {mean_r_manual}')
var_r_manual = 1/(n-1) * (sum_r2_manual - n * mean_r_manual **2)
print (f'var_r_manual {var_r_manual}')
std_r_manual = np.sqrt(var_r_manual)
print (f'std_r_manual {std_r_manual}')
cum_ret_simple_manual = sum_r_manual
print (f'cum_ret_simple_manual {cum_ret_simple_manual}')
cum_ret_cont_manual = np.exp(sum_logret_manual) - 1
print (f'cum_cont_simple_manual {cum_ret_cont_manual}')

In [ ]:
# vectorized python
np.random.seed(0)  # so everyone has the same numbers
n = 100
r = np.random.normal(0.01, 0.05, n)
sum_r_vect = r.sum()
num_r_vect = len(r)
sum_r2_vect = (r**2).sum()
sum_logret_vect = np.log(r+1).sum()
mean_r_vect = sum_r_vect / num_r_vect

print(f'mean_r_vect {mean_r_vect}')
var_r_vect = 1/(n-1) * (sum_r2_vect - n * mean_r_vect**2)
print(f'var_r_vect {var_r_vect}')
std_r_vect = np.sqrt(var_r_vect)
print(f'std_r_vect {std_r_vect}')
cum_ret_simple_vect = sum_r_vect
print(f'cum_ret_simple_vect {cum_ret_simple_vect}')
cum_ret_cont_vect = np.exp(sum_logret_vect) - 1
print(f'cum_ret_cont_vect {cum_ret_cont_vect}')

In [ ]:
# easy numpy
np.random.seed(0)  # so everyone has the same numbers
n = 100
r = np.random.normal(0.01, 0.05, n)
print(f'mean_r {np.mean(r)}')
print(f'var_r {np.var(r, ddof=1)}')
print(f'std_r {np.std(r, ddof=1)}')
print(f'cum_ret_simple {np.sum(r)}')
print(f'cum_ret_cont {np.prod(1+r)-1}')

## 2. A first finance simulation: geometric Brownian motion (discrete)

We simulate *log returns*:
$$
r_t \sim \mathcal{N}(\mu\,\Delta t,\, \sigma\sqrt{\Delta t})
$$
and construct a price path:
$$
S_t = S_0\exp\left(\sum_{i=1}^t r_i\right).
$$

In [ ]:
# simulate GBM
T = 252            # trading days
mu = 0.08           # annual drift (8%)
sigma = 0.20        # annual vol (20%)
dt = 1/T

np.random.seed(0)
log_r = np.random.normal(mu*dt, sigma*np.sqrt(dt), T)
#display(log_r)
#display(np.cumsum(log_r))
S0 = 100.0
S = S0 * np.exp(np.cumsum(log_r))

S[:5], S[-1]

In [ ]:
plt.figure(figsize=(10, 5))

days = np.arange(1, len(S) + 1)
plt.plot(
    days, S,
    color="royalblue",
    linewidth=2.5,
    marker="o",
    markersize=5,
    markerfacecolor="white",
    markeredgewidth=1.2,
    label="Simulated GBM Path"
)

plt.fill_between(days, S, S0, color="royalblue", alpha=0.08)
plt.axhline(S0, color="gray", linestyle="--", linewidth=1, label=f"Initial Price ({S0:.0f})")

plt.title("Simulated Price Path (GBM)", fontsize=14, weight="bold")
plt.xlabel("Day")
plt.ylabel("Price")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# plot GBM
plt.figure()
plt.plot(S)
plt.title("Simulated Price Path (GBM, discrete)")
plt.xlabel("Day")
plt.ylabel("Price")
plt.show()

### Exercise 2 (simulation)

Simulate **5** independent price paths and plot them on the same chart.

**Hint:** store paths in a 2D array of shape `(T, 5)` or `(5, T)`. Use broadcasting carefully.

In [ ]:
# condensed solution
n_paths = 5
T = 252
dt = 1/T
print(dt)
log_r = np.random.normal(mu*dt, sigma*np.sqrt(dt), (T, n_paths))
print(log_r)
S = S0 * np.exp(np.cumsum(log_r, axis=0))
print(np.cumsum(log_r, axis=0))

plt.plot(S)
plt.show()

## 3. pandas basics: Series and DataFrame

pandas is built for **labeled data** (columns + index), especially time series.
- `Series`: 1D labeled array  
- `DataFrame`: 2D labeled table

In [ ]:
# create a pandas series
s = pd.Series([1, 2, 3], name="example")
s

In [ ]:
# create a pandas dataframe
df = pd.DataFrame({"A": [1, 2, 3], "B": [10, 20, 30]})
df

### 3.1 Indexing: `.loc` vs `.iloc`

- `.loc[...]` uses **labels**
- `.iloc[...]` uses **integer positions**

In [ ]:
# how to get to a value in a dataframe
df["A"], df.loc[0], df.iloc[0]

### Exercise 3 (indexing)

Create a DataFrame with a custom index `["a","b","c"]` and columns `x` and `y`.
Then:
- select the row with label `"b"` using `.loc`
- select the second row using `.iloc`
- select column `y`

In [ ]:
# exercise: create and examine a dataframe
d = pd.DataFrame({'x': [1,2,3], 'y': [4,5,6]}, index =['a','b','c'])
display(d)
display(d.loc["b"])
display(d.iloc[1])
display(d["y"])

## 4. Real data: read a CSV from the internet

We'll use an S&P 500 dataset hosted on GitHub.  
You can replace the URL later with any CSV you want.

**Note:** Internet access may be restricted in some classroom environments. If so, download the CSV once and read it from disk.

In [ ]:
# how to read data from the internet 
spx = pd.read_csv("https://cdn.cboe.com/api/global/us_indices/daily_prices/SPX_History.csv")
spx['DATE'] = pd.to_datetime(spx["DATE"])

display(f'SPX history')
display(spx)

spx.set_index("DATE").plot()
plt.grid(True)
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


### 4.1 Parse dates and set an index

## 5. Returns and basic stats

Daily simple returns:
$$
R_t = \frac{S_t}{S_{t-1}} - 1
$$

In [ ]:
# easiest way to get returns
spx["Return"] = spx["SPX"].pct_change()
r = spx["Return"].dropna()
r.head(), r.shape

In [ ]:
# one-liner stats
r.describe()

### Plot price and returns

In [ ]:
# make price plot
plt.figure()
spx["SPX"].plot(title="S&P 500 Level")
plt.show()

plt.figure()
r.plot(title="S&P 500 Daily Returns")
plt.show()

### Exercise 4 (log returns)

Compute log returns:
$$
\ell_t = \ln(S_t) - \ln(S_{t-1})
$$

Compare mean and std of log returns vs simple returns.

In [ ]:
# simple stats of log returns
# log_r = np.log(spx["SP500"]).diff().dropna()
# log_r.describe()